# GUIDE Dataset — Standalone LightGBM Baseline

Target: `IncidentGrade` (TruePositive / BenignPositive / FalsePositive), 3-class classification.

Scope for this notebook:
- Train and evaluate **LightGBM only** — no other models, no ensembling.
- Explicit leakage review before feature selection.
- Group-aware train/validation split keyed on `(OrgId, IncidentId)`, saved to disk so CatBoost/XGBoost notebooks can reuse the exact same split.
- The official `GUIDE_Test.csv` is **not loaded** in this notebook — it stays untouched for a later, unbiased final comparison across models.
- All reported metrics (accuracy, macro-F1, per-class P/R/F1, confusion matrix) are on the held-out **validation** split.

In [ ]:
import os, glob, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

t0 = time.time()
def lap(msg):
    print(f'[{time.time() - t0:7.1f}s] {msg}')

## 1. Load data

Only `GUIDE_Train.csv` is loaded. The official test set is intentionally left alone for now.

In [ ]:
matches = glob.glob('/kaggle/input/**/GUIDE_Train.csv', recursive=True)
if not matches:
    raise FileNotFoundError('GUIDE_Train.csv not found under /kaggle/input')
DATA_DIR = os.path.dirname(matches[0])
lap(f'Using DATA_DIR = {DATA_DIR}')

TARGET = 'IncidentGrade'

df = pd.read_csv(f'{DATA_DIR}/GUIDE_Train.csv')
lap(f'Loaded raw train = {df.shape}')

df = df.drop_duplicates().reset_index(drop=True)
lap(f'After dropping duplicate rows: {df.shape}')

df = df.dropna(subset=[TARGET]).reset_index(drop=True)
lap(f'After dropping rows with missing target: {df.shape}')

assert df[TARGET].notna().all()
print('Class distribution:')
print(df[TARGET].value_counts())

## 2. Leakage review

Before picking features, every column is bucketed explicitly:

- **Identifiers / hashes** (`Id`, `OrgId`, `IncidentId`, `AlertId`, `DeviceId`, `Sha256`, `IpAddress`, `Url`, account/device/application/registry/file identifiers, etc.) — excluded. They are anonymized row/entity keys with no generalizable signal, and `OrgId`/`IncidentId` are reserved below as the **grouping key** for splitting, not as features.
- **Remediation-action columns** (`ActionGrouped`, `ActionGranular`) — **excluded as leakage**. These describe the action a SOC analyst took on the alert, which happens as part of or after triage — i.e. downstream of the grading decision we're trying to predict, not available before it.
- **Verdict-related columns** (`SuspicionLevel`, `LastVerdict`) — reviewed and **kept**. Per the GUIDE dataset's design, these are automated detection-system outputs attached to the alert/entity at ingestion time, generated independently of and prior to the human `IncidentGrade` label — not a restatement of the final grade. This is a documented assumption, not a verified guarantee; flagged as a limitation below.
- Everything else contextual/behavioral (`Category`, `EntityType`, `EvidenceRole`, `ResourceType`, `Roles`, `OSFamily`, `OSVersion`, `AntispamDirection`, `DetectorId`, `AlertTitle`, `CountryCode`, `State`, `City`, `MitreTechniques`, `Timestamp`) — kept, engineered where useful.

In [ ]:
GROUP_COLS = ['OrgId', 'IncidentId']

ID_LIKE_COLS = ['Id', 'AlertId', 'DeviceId', 'Sha256', 'IpAddress', 'Url',
                'AccountSid', 'AccountUpn', 'AccountObjectId', 'AccountName', 'DeviceName',
                'NetworkMessageId', 'EmailClusterId', 'RegistryKey', 'RegistryValueName',
                'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId',
                'ThreatFamily', 'FileName', 'FolderPath', 'ResourceIdName'] + GROUP_COLS

LEAKAGE_COLS = ['ActionGrouped', 'ActionGranular']

CATEGORICAL_COLS = ['Category', 'EntityType', 'EvidenceRole', 'ResourceType', 'Roles',
                     'OSFamily', 'OSVersion', 'AntispamDirection',
                     'SuspicionLevel', 'LastVerdict', 'MitrePrimary']

NUMERIC_COLS = ['DetectorId', 'AlertTitle', 'CountryCode', 'State', 'City',
                'Hour', 'DayOfWeek', 'MitreCount']

print('Excluded as identifiers/hashes:', len(ID_LIKE_COLS), ID_LIKE_COLS)
print('Excluded as remediation-action leakage:', LEAKAGE_COLS)
print('Reviewed verdict columns kept (see note above):', ['SuspicionLevel', 'LastVerdict'])
print('Categorical features kept:', len(CATEGORICAL_COLS), CATEGORICAL_COLS)
print('Numeric features kept:', len(NUMERIC_COLS), NUMERIC_COLS)

## 3. Feature engineering

`Timestamp` → `Hour`/`DayOfWeek`; `MitreTechniques` (semicolon-separated string) → count + primary technique.

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek

mitre = df['MitreTechniques'].fillna('')
df['MitreCount'] = mitre.apply(lambda s: 0 if s == '' else len(s.split(';')))
df['MitrePrimary'] = mitre.apply(lambda s: s.split(';')[0] if s != '' else 'None')

FEATURES = CATEGORICAL_COLS + NUMERIC_COLS
lap(f'Engineered features. Feature count: {len(FEATURES)}')
print(FEATURES)

## 4. Categorical encoding & missing values

Categoricals become pandas `category` dtype (missing values filled with an explicit `'Missing'` category) so LightGBM can use its native categorical split handling instead of treating them as ordinal integers. Numeric columns keep `NaN` as-is — LightGBM handles missing numeric values natively, which is more correct than an arbitrary sentinel value.

In [ ]:
category_maps = {}
for col in CATEGORICAL_COLS:
    df[col] = df[col].fillna('Missing').astype(str).astype('category')
    category_maps[col] = df[col].cat.categories.tolist()

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

target_classes = sorted(df[TARGET].unique())
target_map = {v: i for i, v in enumerate(target_classes)}
inv_target_map = {i: v for v, i in target_map.items()}
df['_target'] = df[TARGET].map(target_map)
print('Target classes:', target_map)

X = df[FEATURES]
y = df['_target']
lap(f'Encoded. X={X.shape}')

## 5. Group-aware, stratified train/validation split

Every row belonging to the same `(OrgId, IncidentId)` pair is forced into the same split, so no incident's alerts are shared between train and validation. `StratifiedGroupKFold` additionally keeps the class balance close to even across the split. The assignment is saved to `split_assignment.csv` keyed by the row's unique `Id`, so the CatBoost/XGBoost notebooks can reuse it exactly by merging on `Id`.

In [ ]:
group_key = df['OrgId'].astype(str) + '_' + df['IncidentId'].astype(str)
n_groups = group_key.nunique()
lap(f'Unique (OrgId, IncidentId) groups: {n_groups}')

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, val_idx = next(sgkf.split(np.zeros(len(df)), y, groups=group_key))

overlap = set(group_key.iloc[train_idx]) & set(group_key.iloc[val_idx])
assert len(overlap) == 0, 'Group leakage between train and validation!'
lap(f'Split sizes -> train={len(train_idx)} ({len(train_idx)/len(df):.1%}), '
    f'val={len(val_idx)} ({len(val_idx)/len(df):.1%}); group overlap = {len(overlap)}')

split_col = pd.Series('train', index=df.index)
split_col.iloc[val_idx] = 'val'
split_assignment = pd.DataFrame({
    'Id': df['Id'], 'OrgId': df['OrgId'], 'IncidentId': df['IncidentId'], 'split': split_col
})
split_assignment.to_csv(f'{OUT_DIR}/split_assignment.csv', index=False)
lap(f'Saved split_assignment.csv ({len(split_assignment)} rows)')

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

## 6. Train LightGBM with early stopping

In [ ]:
n_classes = len(target_map)

model = lgb.LGBMClassifier(
    objective='multiclass', num_class=n_classes,
    n_estimators=1000, num_leaves=63, learning_rate=0.05,
    n_jobs=-1, random_state=RANDOM_STATE
)

train_start = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='multi_logloss',
    categorical_feature=CATEGORICAL_COLS,
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)]
)
training_time = time.time() - train_start
lap(f'Training done in {training_time:.1f}s, best_iteration={model.best_iteration_}')

## 7. Evaluate on validation

In [ ]:
y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average='macro')
weighted_f1 = f1_score(y_val, y_pred, average='weighted')

labels_order = sorted(inv_target_map)
label_names = [inv_target_map[i] for i in labels_order]

report_dict = classification_report(
    y_val, y_pred, labels=labels_order, target_names=label_names, output_dict=True
)
print(classification_report(y_val, y_pred, labels=labels_order, target_names=label_names))
print(f'Accuracy: {accuracy:.4f}  Macro-F1: {macro_f1:.4f}  Weighted-F1: {weighted_f1:.4f}')
print(f'Training time: {training_time:.1f}s')

cm = confusion_matrix(y_val, y_pred, labels=labels_order)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix — LightGBM (validation)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=120)
plt.show()

## 8. Save model, preprocessing info, and results

In [ ]:
joblib.dump(model, f'{OUT_DIR}/lightgbm_model.pkl')

preprocessing_info = {
    'features': FEATURES,
    'categorical_cols': CATEGORICAL_COLS,
    'numeric_cols': NUMERIC_COLS,
    'excluded_id_cols': ID_LIKE_COLS,
    'excluded_leakage_cols': LEAKAGE_COLS,
    'reviewed_kept_cols': ['SuspicionLevel', 'LastVerdict'],
    'group_cols': GROUP_COLS,
    'target_map': target_map,
    'category_values': category_maps,
}
with open(f'{OUT_DIR}/preprocessing_info.json', 'w') as f:
    json.dump(preprocessing_info, f, indent=2)

results = {
    'dataset_rows_after_cleaning': int(len(df)),
    'train_rows': int(len(train_idx)),
    'val_rows': int(len(val_idx)),
    'n_unique_groups': int(n_groups),
    'n_features': len(FEATURES),
    'training_time_seconds': round(training_time, 1),
    'best_iteration': int(model.best_iteration_) if model.best_iteration_ else int(model.n_estimators),
    'accuracy': round(float(accuracy), 4),
    'macro_f1': round(float(macro_f1), 4),
    'weighted_f1': round(float(weighted_f1), 4),
    'per_class_report': report_dict,
    'confusion_matrix': cm.tolist(),
    'target_classes_order': label_names,
}
with open(f'{OUT_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

lap('Saved lightgbm_model.pkl, preprocessing_info.json, results.json, split_assignment.csv')
print(json.dumps({k: v for k, v in results.items() if k not in ('per_class_report', 'confusion_matrix')}, indent=2))

## 9. Summary & limitations

- Dataset: `GUIDE_Train.csv` only, after de-duplication and dropping rows with a missing target.
- Features: 19 total — 11 categorical (native LightGBM categorical handling) + 8 numeric. Pure identifiers, hashes, and the two remediation-action columns were excluded; `SuspicionLevel`/`LastVerdict` were reviewed and kept under a documented assumption (see leakage review above).
- Split: `StratifiedGroupKFold` on `(OrgId, IncidentId)` — no incident's alerts cross the train/validation boundary. Saved to `split_assignment.csv` for reuse by later CatBoost/XGBoost runs.
- The official test set was not touched in this notebook.
- Limitation: the `SuspicionLevel`/`LastVerdict` leakage assessment is a documented assumption based on the dataset's stated design, not something verified against exact event timestamps — worth re-checking if results look suspiciously strong.
- Limitation: `StratifiedGroupKFold` balances classes across groups on a best-effort basis; because group sizes vary, the resulting split isn't exactly 80/20.